In [ ]:
import scipy.sparse as scs
import pandas as pd
import numpy as np

import scanpy as sc
import anndata
from sklearn.cluster import SpectralClustering
import pandas as pd

from scipy.sparse import csr_matrix

import anndata
import numpy as np
import scanpy as sc
import pandas as pd
from sklearn.metrics.cluster import contingency_matrix
from scipy.optimize import linear_sum_assignment
import scipy.sparse as scs
import seaborn as sns


In [ ]:
import yaml
import os.path as op
def read_config(file):
    with open(file, "r") as f:
        config = yaml.safe_load(f)
    return config
pd.set_option('display.float_format', lambda x: '%.3f' % x)


In [1]:
461*461

212521

In [4]:
26287/461


57.02169197396963

In [ ]:
import pandas as pd
import numpy as np


from src.methods.grnboost2.grnboost2_config import GRNBoost2Config
from src.data_simulation.data_simulation_config import DataSimulationConfig
from src.pipelines.utils import PipelineConfig
import os.path as op
import os
from compute_metrics import build_augmented_network
from src.utils import write_config




def parse_args():
    parser = argparse.ArgumentParser(description="Argument parser with two configuration files.")
    
    parser.add_argument(
        '--grnboost2_config',
        type=str,
        default='configuration.json',
        help='Path to the main configuration file (default: configuration.json)'
    )

    parser.add_argument(
        '--dataset_config',
        type=str,
        default='dataset_configuration.json',
        help='Path to the dataset configuration file (default: dataset_configuration.json)'
    )

    parser.add_argument(
        '--pipeline_config',
        type=str,
        default='dataset_configuration.json',
        help='Path to the dataset configuration file (default: dataset_configuration.json)'
    )

    parser.add_argument(
    "--config_list",  # name on the CLI - drop the `--` for positional/required parameters
    nargs="*",  # 0 or more values expected => creates a list
    type=str,
    default=None,  # default if nothing is provided
    )


    args = parser.parse_args()
    config_dict = {}
    for aa in args.config_list:
        n = aa.split('=')
        config_dict[n[0].strip()] = n[1].strip() 
    return args, config_dict




def config_reader(configs):
    config_dict = {}
    for c in configs:
        if  c.startswith('grnboost2'):
            grnboost2_config = GRNBoost2Config.read_yaml(configs[c])
            config_dict[c] = grnboost2_config
    return config_dict



import argparse

args, config_dict = parse_args()
# print(f"Netmap Configuration File: {args.netmap_config}")
# print(f"CSnet Configuration File: {args.csnet_config}")
# print(f"ScGeneRAI Configuration File: {args.scgenerai_config}")
print(f"Dataset Configuration File: {args.dataset_config}")
print(f"Pipeline Configuration File: {args.pipeline_config}")


config_dict = config_reader(config_dict)
dataset_config = DataSimulationConfig.read_yaml(args.dataset_config)
pipeline_config = PipelineConfig.read_yaml(args.pipeline_config)


# read network files
nets = [(op.basename(op.dirname(filename)), pd.read_csv(op.join(pipeline_config.clustered_network_dir, filename), sep=dataset_config.separator)) for filename in dataset_config.edgelist]
off_net = [(op.basename(op.dirname(filename)), pd.read_csv(op.join(pipeline_config.clustered_network_dir, filename), sep=dataset_config.separator)) for filename in dataset_config.common_edges]

print(nets)
# Augmented net contains edges between genes which are controlled by the same transcription factor
augmented_nets = [(net[0], build_augmented_network(net[1])) for net in nets]





collect_results, overlaps = calculate_recovered_edges(grn_ads=grn_ads, nets= nets, augmented_nets=augmented_nets, global_nets=off_net, group_key=dataset_config.group_key)

outdir = op.join(pipeline_config.summary_output_dir, dataset_config.dataset_id)
os.makedirs(outdir, exist_ok = True)
write_config(collect_results, file=op.join(outdir, 'results.yaml'))
overlaps.to_csv(op.join(outdir, 'overlaps.tsv'), sep='\t')


    


In [ ]:
dataset_config = read_config("/data_nfs/og86asub/netmap/netmap-evaluation/results/configurations/data_simulation/config_easy/net_105_55456_net_76_54079_net_51_55147.config.yaml")
net = pd.read_csv('/data_nfs/og86asub/netmap/netmap-evaluation/results/grnboost2/config_2/config_easy/net_105_55456_net_76_54079_net_51_55147//grn.tsv')

nets = [pd.read_csv(op.join("/data_nfs/og86asub/netmap/netmap-evaluation/data/clustered_network/", filename), sep='\t') for filename in dataset_config['edgelist']]
common = [pd.read_csv(op.join("/data_nfs/og86asub/netmap/netmap-evaluation/data/clustered_network/", filename), sep='\t') for filename in dataset_config['common_edges']]


In [137]:
import warnings
warnings.filterwarnings('ignore')
def reformat_dataframe(recovery_rates, config_name):
        # Assuming your DataFrame is named df
    recovery_rates = recovery_rates.pivot_table(
        index=['n_top', 'net'],
        columns='type',
        values='percentage_recovered'
    ).reset_index()
    recovery_rates['method'] = config_name
    recovery_rates = recovery_rates.loc[:, ['method', 'n_top', 'on_target', 'off_target']]
    return recovery_rates



In [138]:
k_thresholds = [1,2,3,4,5,10, 15, 20, 25, 50, 75, 100]
results = compute_metric(net, nets, k_thresholds)


In [140]:
reformat_dataframe(results, 'test')

type,method,n_top,on_target,off_target
0,test,1,0.036,0.018
1,test,1,0.067,0.025
2,test,2,0.045,0.018
3,test,2,0.172,0.043
4,test,3,0.045,0.055
5,test,3,0.209,0.086
6,test,4,0.064,0.073
7,test,4,0.252,0.110
8,test,5,0.082,0.073
9,test,5,0.301,0.123


In [ ]:
def calculate_recovered_edges_per_target(inferred_grn, gold_standard_grn, k_values):
    # Sort inferred edges by score in descending order
    inferred_grn = inferred_grn.sort_values(by='importance', ascending=False).reset_index(drop=True)

    # Create a set of gold standard edges for efficient lookup
    gold_standard_edges = set(zip(gold_standard_grn['source'], gold_standard_grn['target']))
    total_gold_standard_edges = len(gold_standard_edges)

    # Filter k_values to not exceed the total number of inferred edges
    max_k = len(inferred_grn)
    effective_k_values = [k for k in k_values if k <= max_k]
    effective_k_values.append(inferred_grn.shape[0])

    # Initialize a list to store results
    results = []

    for k in effective_k_values:
        
        top_k_inferred = inferred_grn.groupby('target').apply(lambda x: x.nlargest(k, 'importance')).reset_index(drop=True)
        # Create a set of the top k inferred edges
        top_k_edges = set(zip(top_k_inferred['TF'], top_k_inferred['target']))

        # Find the intersection (recovered edges)
        recovered_edges = len(top_k_edges.intersection(gold_standard_edges))

        # Calculate percentage of recovered edges
        if total_gold_standard_edges > 0:
            percentage = (recovered_edges / total_gold_standard_edges)
        else:
            percentage = 0  # Avoid division by zero if gold standard is empty

        results.append({'n_top': k, 'percentage_recovered': percentage})

    results = pd.DataFrame(results)

    return results

In [ ]:
def calculate_recovered_edges(inferred_grn, gold_standard_grn, k_values):
    """
    Computes the percentage of recovered edges for increasing k top edges in an inferred GRN.

    Args:
        inferred_grn (str): Path to a CSV file for the inferred GRN.
                                 The file should have columns: 'regulator', 'target', 'score'.
        gold_standard_grn (str): Path to a CSV file for the gold standard GRN.
                                      The file should have columns: 'regulator', 'target'.
        k_values (list): A list of integers representing the number of top edges to consider.

    Returns:
        pd.DataFrame: A DataFrame with columns 'k' and 'percentage_recovered',
                      showing the recovery percentage for each k value.
    """
    # Sort inferred edges by score in descending order
    inferred_grn = inferred_grn.sort_values(by='importance', ascending=False).reset_index(drop=True)

    # Create a set of gold standard edges for efficient lookup
    gold_standard_edges = set(zip(gold_standard_grn['source'], gold_standard_grn['target']))
    total_gold_standard_edges = len(gold_standard_edges)

    # Filter k_values to not exceed the total number of inferred edges
    max_k = len(inferred_grn)
    effective_k_values = [k for k in k_values if k <= max_k]
    effective_k_values.append(inferred_grn.shape[0])

    # Initialize a list to store results
    results = []

    for k in effective_k_values:
        top_k_inferred = inferred_grn.head(k)

        # Create a set of the top k inferred edges
        top_k_edges = set(zip(top_k_inferred['TF'], top_k_inferred['target']))

        # Find the intersection (recovered edges)
        recovered_edges = len(top_k_edges.intersection(gold_standard_edges))

        # Calculate percentage of recovered edges
        if total_gold_standard_edges > 0:
            percentage = (recovered_edges / total_gold_standard_edges)
        else:
            percentage = 0  # Avoid division by zero if gold standard is empty

        results.append({'n_top': k, 'percentage_recovered': percentage})

    results = pd.DataFrame(results)

    return results

def compute_metric(net, nets, k_thresholds, per_target=True):
    recovery_rates = []
    for grn in net.grn.unique():
        for n in range(len(nets)):
            if per_target:
                recovery_rate = calculate_recovered_edges_per_target( net[net.grn == grn],nets[n], k_thresholds)
            else:
                recovery_rate = calculate_recovered_edges(net[net.grn == grn],nets[n], k_thresholds)
            if grn-1 == n:
                recovery_rate['type'] = 'on_target'
            else:
                recovery_rate['type'] = 'off_target'
            
            recovery_rate['grn'] = int(grn-1)
            recovery_rate['net'] = n
            recovery_rates.append(recovery_rate)
    recovery_rates = pd.concat(recovery_rates)
    return recovery_rates


In [ ]:
sns.lineplot(recovery_rates[recovery_rates.grn==recovery_rates.net], x = 'k', y = 'percentage_recovered', hue='grn')

In [ ]:
sns.lineplot(recovery_rates[recovery_rates.grn!=recovery_rates.net], x = 'k', y = 'percentage_recovered', hue='grn')

In [ ]:



def get_edge_overlap(grn_adata_sub, net, top_n = 150):
    result = {'top_n': top_n}
    result['modules'] = {}
    for g in grn_adata_sub.obs.grn.unique():
        subet = net[(net.grn == g) & (net.grn_effect>0.01)]
        subet['edge' ] = subet['source']+'_'+subet['target']
        subet['edge2' ] = subet['target']+'_'+subet['source']
        augmented = build_augmented_network(subet)

        module_overlap = []
        correct_orientation = []
        false_orientation = []

        for tn in range(10, top_n, 10):
            top_idx = get_top_genes(grn_adata_sub, g, top_n=tn)
            
            module_overlap.append(len(set(augmented.edge).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            correct_orientation.append(len(set(subet.edge).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            false_orientation.append(len(set(subet.edge2).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))



        dicto = {'correct_orientation_count_not_perturbed': correct_orientation,
                  'false_orientation_count_not_perturbed':false_orientation,
                  'module_overlap_not_perturbed': module_overlap}  
        result['modules'][str(g)] = pd.DataFrame(dicto)
        
    result = {'all_edges': result}
    return result


def get_edge_overlap_perturbed_only(grn_adata_sub, net, perturbed_regulators, top_n = 150):
    result = {'top_n': top_n}
    result['modules'] = {}
    for g in grn_adata_sub.obs.grn.unique():
        
        ## Check only the edges that are perturbed.
        subet = net[(net.source.isin(perturbed_regulators[perturbed_regulators.grn==g].disregulated_gene)) & (net.grn == g)]
        subet['edge' ] = subet['source']+'_'+subet['target']
        subet['edge2' ] = subet['target']+'_'+subet['source']
        augmented = build_augmented_network(subet)

        module_overlap = []
        correct_orientation = []
        false_orientation = []

        for tn in range(10, top_n, 10):
            top_idx = get_top_genes(grn_adata_sub, g, top_n=tn)
            
            module_overlap.append(len(set(augmented.edge).intersection(senetmapt(grn_adata_sub.var.index[top_idx].tolist()))))
            correct_orientation.append(len(set(subet.edge).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            false_orientation.append(len(set(subet.edge2).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            
        # Check the edges that are not perturbed
        subet2 = net[(net.grn == g) & (net.grn_effect>0.01)]

        subet2['edge' ] = subet2['source']+'_'+subet2['target']
        subet2['edge2' ] = subet2['target']+'_'+subet2['source']
        augmented2 = build_augmented_network(subet2)

        module_overlap_np = []
        correct_orientation_np = []
        false_orientation_np = []

        for tn in range(10, top_n, 10):
            top_idx = get_top_genes(grn_adata_sub, g, top_n=tn)
            
            module_overlap_np.append(len(set(augmented2.edge).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            correct_orientation_np.append(len(set(subet2.edge).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            false_orientation_np.append(len(set(subet2.edge2).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
        
        
        # Check the edges that should not be there
        subet3 = net[(net.grn == g) & (net.grn_effect<=0.01)]

        subet3['edge' ] = subet3['source']+'_'+subet3['target']
        subet3['edge2' ] = subet3['target']+'_'+subet3['source']
        augmented3 = build_augmented_network(subet3)

        module_overlap_fp = []
        correct_orientation_fp = []
        false_orientation_fp = []

        for tn in range(10, top_n, 10):
            top_idx = get_top_genes(grn_adata_sub, g, top_n=tn)
            
            module_overlap_fp.append(len(set(augmented3.edge).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            correct_orientation_fp.append(len(set(subet3.edge).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            false_orientation_fp.append(len(set(subet3.edge2).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
        
        
        dicto_df = {'correct_orientation_count': correct_orientation,
                  'false_orientation_count':false_orientation,
                  'module_overlap': module_overlap,
                  'correct_orientation_count_not_perturbed': correct_orientation_np,
                  'module_overlap_not_perturbed': module_overlap_np,
                  'false_orienfrom scipy.sparse import csr_matrixtation_count_not_perturbed': false_orientation_np,
                  'correct_orientation_count_fp': correct_orientation_fp,
                  'module_overlap_fp': module_overlap_fp,
                  'false_orientation_fp':false_orientation_fp
                  }
          
        dicto_df = pd.DataFrame(dicto_df)
        dicto_df['p_correct_orientation_count'] = dicto_df['correct_orientation_count']/subet.shape[0]
        dicto_df['p_false_orientation_count'] = dicto_df['false_orientation_count']/subet.shape[0]
        dicto_df['p_correct_orientation_count_not_perturbed'] = dicto_df['correct_orientation_count_not_perturbed']/subet2.shape[0]
        dicto_df['p_false_orientation_count_not_perturbed'] = dicto_df['false_orientation_count_not_perturbed']/subet2.shape[0]
        dicto_df['p_correct_orientation_count_fp'] = dicto_df['correct_orientation_count_fp']/subet3.shape[0]
        dicto_df['p_false_orientation_fp'] = dicto_df['false_orientation_fp']/subet3.shape[0]
        dicto_df['p_module_overlap'] = dicto_df['module_overlap']/augmented.shape[0]
        dicto_df['p_module_overlap_not_perturbed'] = dicto_df['module_overlap_not_perturbed']/augmented2.shape[0]
        dicto_df['p_module_overlap_fp'] = dicto_df['module_overlap_fp']/augmented3.shape[0]

        

        dicto = {'result_df': dicto_df,
                  'perturbed_edges':subet.shape[0],
                  'augmented_edges': augmented.shape[0],
                  'non_perturbed_edges': subet2.shape[0],
                  'augmented_edges_not_perturbed': augmented2.shape[0],
                  'fp_edges': subet3.shape[0],
                  'fp_augmented': augmented3.shape[0]
                  }  
        

        result['modules'][str(g)] = dicto
    
    
    result = {'perturbed_only': result}
    return result
    


def get_edgelist(grn_adata_sub, grn_varname = 'MAPK1_NegCtrl0', cluster_var='spectral_remap', top_n  = 150):
    top_idx = get_top_genes(grn_adata_sub, grn_varname, cluster_var=cluster_var, top_n=top_n)
    edgelist = grn_adata_sub.var.iloc[top_idx]
    edgelist = edgelist.reset_index()
    edgelist[['source', 'target']] = edgelist['index'].str.split('_', expand = True)
    return edgelist





def analysis(aac, nlc, net, top_genes=500):
    # Check if aac is sparse; convert to dense if necessary for operations like nansum
    if isinstance(aac, csr_matrix):
        sumi = np.array(aac.sum(axis=0)).flatten()  # Efficient summation for sparse
    else:
        sumi = np.array(np.nansum(aac, axis=0)).flatten()  # Fallback for dense
    
    sumia = np.flip(np.argsort(sumi))
    nlc = np.array(nlc)
    nlc = pd.DataFrame(nlc)
    nlc.columns = ['source', 'target']
    
    fraction_of_regulators_after_edge_selection = []
    for th in range(10, top_genes, 10):
        high = nlc.iloc[sumia[0:th]]
        high = pd.DataFrame(high)
        high.columns = ['source', 'target']
        rol = overrepresentation(edgelist=high)
        fraction_of_regulators_after_edge_selection.append(rol[rol.index.isin(net.source.unique())]['percentage'].sum())
    
    return fraction_of_regulators_after_edge_selection


    
def overrepresentation(edgelist, source = 'source', target = 'target'):
    occ = edgelist[source].value_counts().add(edgelist[target].value_counts(), fill_value=0)
    occ = pd.DataFrame(occ)
    occ = occ.sort_values('count', ascending=False)
    occ['percentage'] = occ['count']/occ['count'].sum()
    return occ


In [ ]:



def random_score_distributions(grn_adata, percentile = 95, column_of_interest = 'spectral', rel_abs = False):
    """
    Create random score distributions across clusters via subsampling
    Compare the score of the edges within one cluster and select only
    those which fill above the p percentile of the randomized scores.
    Return the indices of those edges.
    
    """
    if rel_abs:
        grn_adata.X = np.abs(grn_adata.X)
    relevant_indices = {}
    
    for c in grn_adata.obs[column_of_interest].unique():
        background_sum_distribution = []
        for i in range(1000):
            samples = len(np.where(grn_adata.obs[column_of_interest] == c)[0])
            rs = np.random.choice( range(grn_adata.obs.shape[0]),size=samples, replace=False)
            sum_of_lrps = grn_adata.X[rs, :].sum(axis = 0)
            background_sum_distribution.append(sum_of_lrps)
        background_sum_distribution = np.stack(background_sum_distribution)
        background_sum_distribution = np.array(background_sum_distribution)
        percentiles = np.percentile(background_sum_distribution, axis = 0, q = percentile)
    
        sum_of_interest = grn_adata.X[grn_adata.obs[column_of_interest] == c, :].sum(axis = 0)
        #sum_of_interest = np.array(sum_of_interest).reshape(sum_of_interest.shape[1])
        relevant_indices[c] = np.where((sum_of_interest-percentiles)>0)[0]
    return relevant_indices






def create_cluster_subset(grn_adata, grn_adata_sub, rel, cluster=0, obs = 'spectral', add_obs = ['spectral_remap']):
    ## subset only edges relevant in this cluster from original object
    sub1 = grn_adata[:, rel[int(cluster)]]
    #add the metadata from the remapped subset
    sub1.obs[[obs]+add_obs] = grn_adata_sub.obs[[obs]+add_obs] 
    # subset the correct group
    sub1 = sub1[sub1.obs[obs]==cluster]
    return sub1


def get_edges(sub1):
    sub1.var[['source','target']] =[x.split('_') for x in sub1.var.index.tolist() ]
    edges = scs.find(sub1.X)
    barcode = [sub1.obs.index[i] for i in edges[0]]
    source = [sub1.var.source[i] for i in edges[1]]
    target = [sub1.var.target[i] for i in edges[1]]
    edge = [sub1.var.index[i] for i in edges[1]]
    edgedf = pd.DataFrame({'barcode': barcode, 'edge': edge, 'source': source, 'target': target,  'value': edges[2]})
    
    summary = edgedf.groupby('edge').median('value')
    summary['mean'] = edgedf.groupby('edge').mean('value')
    summary = summary.rename(columns= {'value': 'median'})
    summary = summary.sort_values('median', ascending=False)
    return summary

def create_all_summaries(grn_adata, grn_adata_sub, rel, cluster_col = 'spectral'):
    summaries = []
    for c in np.unique(grn_adata_sub.obs[cluster_col]):
        sub1 = create_cluster_subset(grn_adata, grn_adata_sub, rel, cluster = c)
        summary = get_edges(sub1)
        summary[cluster_col] = c
        summaries.append(summary)
    summaries = pd.concat(summaries)
    return summaries



In [ ]:
def split_index(aa):    
    aa.var['source']   = [l[0] for l in aa.var.index.str.split('_', expand=True)]
    aa.var['target']   = [l[1] for l in aa.var.index.str.split('_', expand=True)]
    return aa

In [ ]:
scgenerai = sc.read_h5ad('/data_nfs/og86asub/netmap/netmap-evaluation/results/scgenerai/config/config_easy/net_51_43266_net_82_42088_net_105_43582/grn_lrp.h5ad')
csnet = sc.read_h5ad('/data_nfs/og86asub/netmap/netmap-evaluation/results/csnet/config/config_easy/net_51_43266_net_82_42088_net_105_43582/csnet.csn.h5ad')
netmap = sc.read_h5ad('/data_nfs/og86asub/netmap/netmap-evaluation/results/netmap/config/config_easy/net_51_43266_net_82_42088_net_105_43582/grn_lrp.h5ad')

In [ ]:
csnet = csnet[:, csnet.layers['significance'].sum(axis = 0)!= 0].copy()
csnet.var = csnet.var.set_index('edge')
csnet = split_index(csnet)

In [ ]:
scgenerai.var

In [ ]:
scgenerai = split_index(scgenerai)

In [ ]:
net = pd.read_csv('/data_nfs/og86asub/netmap/netmap-evaluation/data/clustered_network/net_82_42088/edges.tsv', sep= '\t')

In [ ]:
net2 = pd.read_csv('/data_nfs/og86asub/netmap/netmap-evaluation/data/clustered_network/net_105_43582/edges.tsv', sep= '\t')

In [ ]:
collect_results
